# ETL

In [ ]:
import os
import warnings

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from botocore.client import Config
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'oiluser')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'oilpass')}@"
    f"{PG_HOST}:{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'oildb')}"
)
engine = create_engine(PG_URL)

def minio_endpoint_url():
    raw = os.getenv("MINIO_ENDPOINT", "localhost:9000").strip().rstrip("/")
    if raw.startswith("http://") or raw.startswith("https://"):
        return raw
    return f"http://{raw}"

def minio_client():
    return boto3.client(
        "s3",
        endpoint_url=minio_endpoint_url(),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )


In [ ]:

import subprocess, sys
etl_path = "/home/jovyan/work/etl/export_to_minio.py"
if os.path.exists(etl_path):
    subprocess.run([sys.executable, etl_path], check=False)

client = minio_client()
buckets = client.list_buckets()
print("MinIO buckets:", [b["Name"] for b in buckets["Buckets"]])



In [ ]:

production = pd.read_sql("SELECT * FROM production", engine)
wells = pd.read_sql("SELECT * FROM wells", engine)
df = production.merge(wells, on="well_id")

for col in ["temperature", "pressure"]:
    df.loc[df["oil_ton"] > 0, col] = df.loc[df["oil_ton"] > 0, col].fillna(df[col].median())

active = df[df["oil_ton"] > 0]
q1, q3 = active["oil_ton"].quantile([0.25, 0.75])
iqr = q3 - q1
df = df[~((df["oil_ton"] > 0) & ((df["oil_ton"] < q1 - 1.5*iqr) | (df["oil_ton"] > q3 + 1.5*iqr)))]

df["downtime_ratio"] = (df["downtime_hours"].fillna(0) / 24).clip(0, 1)
df["avg_pressure"] = df["pressure"]
df["avg_temperature"] = df["temperature"]

daily = df.groupby("date").agg(
    total_oil=("oil_ton", "sum"),
    avg_pressure=("pressure", "mean"),
    avg_temperature=("temperature", "mean"),
    downtime_coef=("downtime_ratio", "mean"),
).reset_index()

well_agg = df[df["oil_ton"] > 0].groupby(["well_id", "name"]).agg(
    avg_oil=("oil_ton", "mean"),
    downtime_pct=("downtime_ratio", "mean"),
).reset_index()
well_agg["downtime_pct"] *= 100

print("Daily aggregate:\n", daily.head())
print("\nWell KPI:\n", well_agg)

